# Fig S2 — allele flips: kriging followed by SuSiE-RSS or SuSiE-rRSS (data processing)

Same simulations as Fig 1B (GTEx, in-sample LD, 0.1% / 1% allele-flip rates, 1–3 causal variants, NCP 40 / 80 / 120). Compares uncorrected SuSiE-RSS, kriging + SuSiE-RSS (`02` `[krigging_fix]`, `Fine_Flip`) and kriging + the full SuSiE-rRSS pipeline (`25` `[krigging_fix_rrss]`, `Fine_Flip_rRSS`; the SER model replaces the fit when `R_reliability_flag` is set). Uncorrected SuSiE-rRSS is kept in the data as well. Both post-QC fits use the same kriging-corrected z.

**Outputs:**
- `figure_data/FigS2.Allele_Flip_Kriging_SuSiE_rRSS.rds` — power / FDR with bootstrap 95% CIs
- `figure_data/FigS2.Allele_Flip_Kriging_SuSiE_rRSS_SER_Fallback.rds` (+ `.csv`) — SER fallback per setting (flip rate × NCP × causal variants) for SuSiE-rRSS without allele QC and after kriging

In [1]:
source("../../R/utils.R")

data_name <- "FigS2.Allele_Flip_Kriging_SuSiE_rRSS"
n_gwas    <- 800                      # GTEx eQTL sample size; NCP = n * h2g

rss  <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Fine_Flip/summary/result_susie_bias_summary_summary.rds")
rrss <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Fine_Flip_rRSS/summary/result_susie_bias_summary_summary.rds")

In [2]:
## ---- methods ----------------------------------------------------------------------
## SuSiE-RSS (02 [krigging_fix], as in Fig 1B): uncorrected and after kriging
rss_rows <- rss %>% filter(method %in% c("flipped_res", "kriging_res"))

## full SuSiE-rRSS pipeline (25 [krigging_fix_rrss]): the fit, or its SER model when R_reliability_flag is set
full_pipeline <- function(fit_name) {
  fit <- filter(rrss, method == fit_name)
  ser <- filter(rrss, method == paste0(fit_name, "_SER"))
  use_ser <- fit %>% filter(R_reliability_flag %in% TRUE) %>% select(input_file) %>% semi_join(ser, by = "input_file")
  bind_rows(ser %>% semi_join(use_ser, by = "input_file"),
            fit %>% anti_join(use_ser, by = "input_file")) %>%
    mutate(method = paste0(fit_name, "_full"))
}
rrss_rows <- bind_rows(full_pipeline("kriging_rrss"), full_pipeline("flipped_rrss"))

## SER fallback frequency of the SuSiE-rRSS fits
rrss %>% filter(method %in% c("kriging_rrss", "flipped_rrss")) %>%
  mutate(miss_prop = str_extract(input_file, "miss_prop_[0-9.]+")) %>%
  group_by(method, miss_prop) %>%
  summarise(n = n(), n_fallback = sum(R_reliability_flag %in% TRUE),
            pct = round(100 * mean(R_reliability_flag %in% TRUE), 1), .groups = "drop")

qc_lab <- c(flipped_res        = "SuSiE-RSS",
            kriging_res        = "Kriging + SuSiE-RSS",
            kriging_rrss_full  = "Kriging + SuSiE-rRSS",
            flipped_rrss_full  = "SuSiE-rRSS")

cell_reps <- bind_rows(rss_rows, rrss_rows) %>%
  mutate(qc        = qc_lab[method],
         h2g       = str_extract(input_file, "h2g_[0-9.]+"),
         miss_prop = str_extract(input_file, "miss_prop_[0-9.]+"),
         causal_k  = total_var_number) %>%
  filter(!is.na(qc))

method,miss_prop,n,n_fallback,pct
<chr>,<chr>,<int>,<int>,<dbl>
flipped_rrss,miss_prop_0.001,4500,3286,73.0
flipped_rrss,miss_prop_0.01,4500,4013,89.2
flipped_rrss,miss_prop_0.1,4500,4449,98.9
kriging_rrss,miss_prop_0.001,4500,27,0.6
kriging_rrss,miss_prop_0.01,4500,179,4.0
kriging_rrss,miss_prop_0.1,4500,4479,99.5


In [3]:
## ---- bootstrap over replicates within each cell (boot_cell() in R/utils.R) --------
set.seed(1)
summary_ci <- cell_reps %>%
  group_by(qc, h2g, miss_prop, total_var_number) %>%
  group_modify(~ boot_cell(.x, B = 1000)) %>%
  ungroup()

plot_df <- summary_ci %>%
  mutate(
    miss_prop_num = as.numeric(str_remove(miss_prop, "miss_prop_")),
    miss_prop = fct_reorder(paste0("Flip Rate ", formatC(miss_prop_num * 100, format = "g"), "%"), miss_prop_num),
    h2g_num = as.numeric(str_remove(h2g, "h2g_")),
    ncp_num = n_gwas * h2g_num,
    ncp = fct_reorder(paste0("NCP = ", formatC(ncp_num, format = "f", digits = 1)), ncp_num),
    qc = factor(qc, levels = unname(qc_lab))
  ) %>%
  filter(miss_prop_num != 0.1)                   # 0.1% and 1% flip rates (Fig S2)

save_data(plot_df, data_name)

In [4]:
## ---- ranges across the 18 settings (Results l.195-197) ---------------------------
plot_df %>%
  group_by(qc) %>%
  summarise(FDR_min = round(100 * min(FDR, na.rm = TRUE), 1), FDR_max = round(100 * max(FDR, na.rm = TRUE), 1),
            power_min = round(100 * min(power), 1), power_max = round(100 * max(power), 1), .groups = "drop")
## the text quotes 3.3-16.5% FDR and 88.9-97.3% power for Kriging + SuSiE-RSS; check that row reproduces them

qc,FDR_min,FDR_max,power_min,power_max
<fct>,<dbl>,<dbl>,<dbl>,<dbl>
SuSiE-RSS,59.9,89.4,10.9,46.2
Kriging + SuSiE-RSS,3.3,16.5,88.9,97.3
Kriging + SuSiE-rRSS,2.5,6.1,82.9,97.4
SuSiE-rRSS,5.3,25.5,18.3,93.6


## SER fallback per setting

Share of SuSiE-rRSS analyses whose reliability flag sends them to the SER model, for each flip rate × NCP × number of causal variants, without allele QC and after kriging. Denominator: all analyses in the setting, including those without a reported credible set. Labels match `plot_df`.

In [5]:
ser_df <- rrss %>%
  filter(method %in% c("flipped_rrss", "kriging_rrss")) %>%
  mutate(qc            = qc_lab[paste0(method, "_full")],
         miss_prop_num = as.numeric(str_remove(str_extract(input_file, "miss_prop_[0-9.]+"), "miss_prop_")),
         ncp_num       = n_gwas * as.numeric(str_remove(str_extract(input_file, "h2g_[0-9.]+"), "h2g_"))) %>%
  filter(miss_prop_num != 0.1) %>%                # 0.1% and 1% flip rates, as plot_df
  group_by(qc, miss_prop_num, ncp_num, total_var_number) %>%
  summarise(n = n(), n_fallback = sum(R_reliability_flag %in% TRUE), .groups = "drop") %>%
  mutate(fallback  = n_fallback / n,
         miss_prop = fct_reorder(paste0("Flip Rate ", formatC(miss_prop_num * 100, format = "g"), "%"), miss_prop_num),
         ncp       = fct_reorder(paste0("NCP = ", formatC(ncp_num, format = "f", digits = 1)), ncp_num),
         qc        = factor(qc, levels = c("SuSiE-rRSS", "Kriging + SuSiE-rRSS")))

## pooled over settings this reproduces the table above (0.1% / 1% rows)
ser_df %>% group_by(qc, miss_prop) %>% summarise(n = sum(n), n_fallback = sum(n_fallback), .groups = "drop") %>%
  mutate(pct = round(100 * n_fallback / n, 1))
ser_df %>% transmute(qc, miss_prop, ncp, causal = total_var_number, fallback = sprintf("%d/%d (%.1f%%)", n_fallback, n, 100 * fallback)) %>% as.data.frame()

save_data(ser_df, paste0(data_name, "_SER_Fallback"))

qc,miss_prop,n,n_fallback,pct
<fct>,<fct>,<int>,<int>,<dbl>
SuSiE-rRSS,Flip Rate 0.1%,4500,3286,73.0
SuSiE-rRSS,Flip Rate 1%,4500,4013,89.2
Kriging + SuSiE-rRSS,Flip Rate 0.1%,4500,27,0.6
Kriging + SuSiE-rRSS,Flip Rate 1%,4500,179,4.0


qc,miss_prop,ncp,causal,fallback
<fct>,<fct>,<fct>,<int>,<chr>
Kriging + SuSiE-rRSS,Flip Rate 0.1%,NCP = 40.0,1,4/500 (0.8%)
Kriging + SuSiE-rRSS,Flip Rate 0.1%,NCP = 40.0,2,4/500 (0.8%)
Kriging + SuSiE-rRSS,Flip Rate 0.1%,NCP = 40.0,3,3/500 (0.6%)
Kriging + SuSiE-rRSS,Flip Rate 0.1%,NCP = 80.0,1,1/500 (0.2%)
Kriging + SuSiE-rRSS,Flip Rate 0.1%,NCP = 80.0,2,6/500 (1.2%)
Kriging + SuSiE-rRSS,Flip Rate 0.1%,NCP = 80.0,3,1/500 (0.2%)
Kriging + SuSiE-rRSS,Flip Rate 0.1%,NCP = 120.0,1,4/500 (0.8%)
Kriging + SuSiE-rRSS,Flip Rate 0.1%,NCP = 120.0,2,3/500 (0.6%)
Kriging + SuSiE-rRSS,Flip Rate 0.1%,NCP = 120.0,3,1/500 (0.2%)
